# LightGBM liviano con validacion cruzada y metricas completas

Replica el flujo de `11_xgboost_v3.ipynb` (lectura de variantes, balanceo, CV y reportes CSV/PDF) pero usando LightGBM con hiperparametros mas livianos para poder recorrer ~57 variantes sin tiempos excesivos.

In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from matplotlib.backends.backend_pdf import PdfPages
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [2]:
# === CONFIGURACION ===
INPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23'
BASE_OUTPUT_PATH = r'c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/models'
MODEL_NAME = 'LightGBM'
INTENTO = 5  # Cambiar si generas una nueva version
N_SPLITS = 3  # Menos folds para acelerar con muchas variantes
N_ESTIMATORS = 180
LEARNING_RATE = 0.05

fecha_actual = datetime.today().strftime('%Y-%m-%d')
OUTPUT_PATH = os.path.join(BASE_OUTPUT_PATH, f'{MODEL_NAME}_{INTENTO:02d}_{fecha_actual}')
os.makedirs(OUTPUT_PATH, exist_ok=True)

BALANCE_METHODS = {
    'NONE': None,
    'SMOTE': SMOTE(random_state=42),
    'UNDER': RandomUnderSampler(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42)
}

metricas_totales = []

# Detectar variantes de datasets
variantes_X = sorted([f for f in os.listdir(INPUT_PATH) if f.startswith('X_train')])
print(f'Se detectaron {len(variantes_X)} variantes de X_train_* en {INPUT_PATH}')


def build_lgbm(num_classes: int):
    'Modelo liviano para multi/binary clase.'
    params = dict(
        boosting_type='gbdt',
        objective='multiclass' if num_classes > 2 else 'binary',
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        num_leaves=31,
        max_depth=-1,
        subsample=0.85,
        colsample_bytree=0.9,
        max_bin=63,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        bagging_freq=1,
        n_jobs=-1,
        random_state=42,
    )
    if num_classes > 2:
        params['num_class'] = num_classes
    return LGBMClassifier(**params)


Se detectaron 59 variantes de X_train_* en c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/intermediate/selected/04_2025_08_23


In [3]:
for balance_name, balancer in BALANCE_METHODS.items():
    print(f"=== Balanceo: {balance_name} ===")
    output_subdir = os.path.join(OUTPUT_PATH, balance_name)
    os.makedirs(output_subdir, exist_ok=True)

    for x_file in variantes_X:
        base_name = x_file.replace('X_train_', '').replace('.parquet', '')
        try:
            print(f' Procesando: {base_name}')

            # Cargar datos
            X_train = pd.read_parquet(os.path.join(INPUT_PATH, f'X_train_{base_name}.parquet'))
            X_test = pd.read_parquet(os.path.join(INPUT_PATH, f'X_test_{base_name}.parquet'))
            y_train = pd.read_parquet(os.path.join(INPUT_PATH, f'y_train_{base_name}.parquet')).squeeze()
            y_test = pd.read_parquet(os.path.join(INPUT_PATH, f'y_test_{base_name}.parquet')).squeeze()

            # Ajustar etiquetas para que arranquen en 0
            y_train = y_train - y_train.min()
            y_test = y_test - y_test.min()

            # Balanceo solo en train
            if balancer is not None:
                X_train_bal, y_train_bal = balancer.fit_resample(X_train, y_train)
            else:
                X_train_bal, y_train_bal = X_train, y_train

            num_classes = int(pd.Series(y_train_bal).nunique())
            if num_classes < 2:
                print('  Dataset con una sola clase; se omite.')
                continue

            # Validacion cruzada rapida sobre el train balanceado
            cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
            model_cv = build_lgbm(num_classes)
            cv_scores = cross_val_score(model_cv, X_train_bal, y_train_bal, cv=cv, scoring='f1_macro')
            cv_f1_mean = float(np.mean(cv_scores))
            cv_f1_std = float(np.std(cv_scores))

            # Entrenamiento
            start = time.time()
            model = build_lgbm(num_classes)
            model.fit(X_train_bal, y_train_bal)
            tiempo = (time.time() - start) / 60

            # Prediccion
            y_pred_train = model.predict(X_train_bal)
            y_pred_test = model.predict(X_test)
            y_proba = model.predict_proba(X_test)

            # Reporte completo
            report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True)).T
            report_train = pd.DataFrame(classification_report(y_train_bal, y_pred_train, output_dict=True)).T
            report_test['set'] = 'test'
            report_train['set'] = 'train'
            full_report = pd.concat([report_train, report_test])
            full_report['tiempo_min'] = tiempo

            # Guardar CSV de metricas por clase
            full_report.to_csv(os.path.join(output_subdir, f'metricas_{base_name}.csv'))

            # Matriz de confusion
            cm = confusion_matrix(y_test, y_pred_test)

            # Guardar PDF con visualizaciones
            with PdfPages(os.path.join(output_subdir, f'reporte_{base_name}.pdf')) as pdf:
                # Confusion Matrix
                ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap='Blues')
                plt.title('Matriz de Confusion')
                pdf.savefig(); plt.close()

                # ROC por clase
                classes = np.unique(y_test)
                y_bin = label_binarize(y_test, classes=classes)
                plt.figure()
                for i, clase in enumerate(classes):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
                    roc_auc = auc(fpr, tpr)
                    plt.plot(fpr, tpr, label=f'Clase {clase} (AUC={roc_auc:.2f})')
                plt.plot([0, 1], [0, 1], 'k--')
                plt.title('Curvas ROC por clase')
                plt.xlabel('FPR')
                plt.ylabel('TPR')
                plt.legend()
                pdf.savefig(); plt.close()

                # PR Curve por clase
                plt.figure()
                for i, clase in enumerate(classes):
                    precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
                    pr_auc = average_precision_score(y_bin[:, i], y_proba[:, i])
                    plt.plot(recall, precision, label=f'Clase {clase} (PR AUC={pr_auc:.2f})')
                plt.title('Curvas Precision-Recall por clase')
                plt.xlabel('Recall')
                plt.ylabel('Precision')
                plt.legend()
                pdf.savefig(); plt.close()

            # Resumen final para comparabilidad
            resumen = {
                'modelo': base_name,
                'balanceo': balance_name,
                'accuracy_test': accuracy_score(y_test, y_pred_test),
                'macro_f1_test': report_test.loc['macro avg', 'f1-score'],
                'weighted_f1_test': report_test.loc['weighted avg', 'f1-score'],
                'accuracy_train': accuracy_score(y_train_bal, y_pred_train),
                'macro_f1_train': report_train.loc['macro avg', 'f1-score'],
                'weighted_f1_train': report_train.loc['weighted avg', 'f1-score'],
                'tiempo_min': tiempo,
                'sobreajuste_macro_f1': report_train.loc['macro avg', 'f1-score'] - report_test.loc['macro avg', 'f1-score'],
                'cv_macro_f1_mean': cv_f1_mean,
                'cv_macro_f1_std': cv_f1_std
            }
            for clase in report_test.index:
                if clase not in ['accuracy', 'macro avg', 'weighted avg']:
                    resumen[f'f1_{clase}_test'] = report_test.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_test'] = report_test.loc[clase, 'recall']
                    resumen[f'precision_{clase}_test'] = report_test.loc[clase, 'precision']
                    resumen[f'f1_{clase}_train'] = report_train.loc[clase, 'f1-score']
                    resumen[f'recall_{clase}_train'] = report_train.loc[clase, 'recall']
                    resumen[f'precision_{clase}_train'] = report_train.loc[clase, 'precision']
            metricas_totales.append(resumen)

            print(f" {base_name} [{balance_name}]: F1_clase_1_test={resumen.get('f1_1_test', 0):.3f}, Recall_clase_1_test={resumen.get('recall_1_test', 0):.3f}, Tiempo={tiempo:.2f}min")

        except Exception as e:
            print(f" Error en {base_name} con balanceo {balance_name}: {e}")


=== Balanceo: NONE ===
 Procesando: MaxAbs_FE_ALL
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.801453 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14701
[LightGBM] [Info] Number of data points in the train set: 298925, number of used features: 1140
[LightGBM] [Info] Start training from score -2.379916
[LightGBM] [Info] Start training from score -1.639819
[LightGBM] [Info] Start training from score -1.547894
[LightGBM] [Info] Start training from score -1.498879
[LightGBM] [Info] Start training from score -1.282473
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 wi

In [4]:
# Consolidar todas las metricas en un CSV unico
if metricas_totales:
    df_metricas = pd.DataFrame(metricas_totales)
    df_metricas.to_csv(os.path.join(OUTPUT_PATH, 'metricas_totales.csv'), index=False)
    print(f'Se guardo metricas_totales.csv con {len(df_metricas)} filas en {OUTPUT_PATH}')
    display(df_metricas.head())
else:
    print('No se genero ninguna metrica; revisa los logs de arriba.')


Se guardo metricas_totales.csv con 236 filas en c:/Users/Gonzalo/Documents/GitHub/TesisAustral/Archivos_tesis/models\LightGBM_05_2025-12-28


,modelo,balanceo,accuracy_test,macro_f1_test,weighted_f1_test,accuracy_train,macro_f1_train,weighted_f1_train,tiempo_min,sobreajuste_macro_f1,...,precision_3_test,f1_3_train,recall_3_train,precision_3_train,f1_4_test,recall_4_test,precision_4_test,f1_4_train,recall_4_train,precision_4_train
0,MaxAbs_FE_ALL,NONE,0.516111,0.401704,0.459133,0.526729,0.413278,0.471283,2.436636,0.011574,...,0.398170,0.464216,0.529118,0.413496,0.774605,0.886109,0.688028,0.773868,0.885495,0.687234
1,MaxAbs_FE_ANOVA,NONE,0.378472,0.241367,0.287191,0.380220,0.245053,0.291006,0.171283,0.003685,...,0.265875,0.173773,0.127764,0.271566,0.571504,0.804960,0.443018,0.571293,0.802374,0.443551
2,MaxAbs_FE_MI,NONE,0.427287,0.295581,0.349760,0.434204,0.304050,0.358789,0.288356,0.008469,...,0.382077,0.326388,0.278002,0.395166,0.630995,0.829147,0.509285,0.633362,0.830091,0.512015
3,MaxAbs_FE_PCA30,NONE,0.462497,0.344196,0.401981,0.481123,0.365125,0.424246,0.444788,0.020929,...,0.372401,0.415802,0.427282,0.404922,0.677335,0.847448,0.564100,0.680218,0.847911,0.567902
4,MaxAbs_FE_PCAopt,NONE,0.504567,0.398096,0.455524,0.533658,0.430197,0.488823,6.378689,0.032101,...,0.393031,0.470179,0.510059,0.436083,0.755812,0.869544,0.668389,0.761688,0.872492,0.675856
